# Random Forest Training Notebook

This notebook trains a Random Forest on the preprocessed CICEVSE2024 dataset for Multiclass classification. It includes data loading, visualization, evaluation, and hyperparameter tuning using `GridSearchCV`.

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Ensure plots are displayed inline
%matplotlib inline

## 1. Load Data
Load the preprocessed train, validation, and test datasets. Note: ensure `preprocess.py` has been run previously.

In [ ]:
# Adjust path assuming the notebook runs from the project root or src/models/random_forest
import sys
if os.path.exists('../../../data/processed'):
    DATA_DIR = '../../../data/processed'
elif os.path.exists('data/processed'):
    DATA_DIR = 'data/processed'
else:
    DATA_DIR = '../data/processed' # fallback

print(f"Using data directory: {DATA_DIR}")

X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))
y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))

y_train_multi = y_train["Label_Multiclass"].values.ravel()
y_val_multi = y_val["Label_Multiclass"].values.ravel()
y_test_multi = y_test["Label_Multiclass"].values.ravel()

print("Data loaded successfully!")
print(f"X_train shape: {X_train.shape}")

## 1.5. Visualize Class Distributions
Before training the model, let's visualize the class distributions of our training dataset for Multiclass targets.

In [ ]:
plt.figure(figsize=(10, 5))

# Plot Multiclass Distribution
sns.countplot(y="Label_Multiclass", data=y_train, order=y_train["Label_Multiclass"].value_counts().index)
plt.title("Multiclass Distribution")
plt.xlabel("Count")
plt.ylabel("Label_Multiclass")

plt.tight_layout()
plt.show()

## 2. Hyperparameter Tuning (Multiclass Classification)
We use `RandomForestClassifier`. We'll tune the `n_estimators` and `max_depth` parameters using cross-validation on the training set.

In [ ]:
print("Starting Hyperparameter Tuning for Multiclass Model...")
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, None]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'), 
                        param_grid, 
                        cv=3, 
                        scoring='f1_macro', 
                        n_jobs=-1, 
                        verbose=2)

rf_grid.fit(X_train, y_train_multi)

print(f"Best Parameters: {rf_grid.best_params_}")
print(f"Best CV F1-Score (Macro): {rf_grid.best_score_:.4f}")

best_multi_model = rf_grid.best_estimator_

## 3. Evaluation Setup

In [ ]:
def evaluate_model(model, X, y, title_prefix="", is_multiclass=False):
    preds = model.predict(X)
    avg_method = 'weighted' if is_multiclass else 'binary'
    
    print(f"--- {title_prefix} Classification Report ---")
    print(classification_report(y, preds, zero_division=0))
    
    cm = confusion_matrix(y, preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{title_prefix} Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    return preds

## 4. Evaluate Multiclass Model

In [ ]:
_ = evaluate_model(best_multi_model, X_val, y_val_multi, title_prefix="Multiclass Validation", is_multiclass=True)
_ = evaluate_model(best_multi_model, X_test, y_test_multi, title_prefix="Multiclass Test", is_multiclass=True)

## 5. Save Model
Export the best multiclass model to the `saved_models` directory.

In [ ]:
if os.path.exists('../../../saved_models'):
    SAVE_DIR = '../../../saved_models'
elif os.path.exists('saved_models'):
    SAVE_DIR = 'saved_models'
else:
    SAVE_DIR = '../saved_models'

os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(best_multi_model, os.path.join(SAVE_DIR, "rf_model_multiclass.pkl"))

print("Model saved successfully!")